# GPU Performance Prediction — Rating Standard Model (std.ipynb)

本 Notebook 实现了评分标准模型，用于评估参赛者提交的预测质量。

**模型策略：**
1. 丰富的特征工程（带宽延迟、计算延迟、对数变换、比率特征）
2. XGBoost 多输出回归
3. 5折交叉验证评估泛化能力
4. 分场景特征处理

**适用条件：**
- 需要先运行 `dataset/dataset-process.py` 和 `dataset/dataset-split.py` 生成数据
- 训练时间：T4 GPU上 < 10分钟
- 参数量：< 10^6

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score

from xgboost import XGBRegressor

print('Libraries imported.')

## 1. 加载与探索数据

In [ ]:
DATA_DIR = 'train.csv'

df_all = pd.read_csv(DATA_DIR)
print(f'All processed data: {df_all.shape}')
print(f'Columns: {list(df_all.columns)}')

In [ ]:
TARGET_COLS = [
    'gpu_power_draw_watts',
    'avg_e2e_latency_seconds',
    'energy_efficiency_tokens_per_joule',
    'throughput_tokens_per_second',
]

print('Target distributions:')
print(df_all[TARGET_COLS].describe().round(4))

print('\nTarget correlations:')
print(df_all[TARGET_COLS].corr().round(3))

In [ ]:
if 'scenario' in df_all.columns:
    print('Scenario distribution:')
    print(df_all['scenario'].value_counts())

if 'model' in df_all.columns and 'gpu_type' in df_all.columns:
    print(f'\nUnique models: {df_all["model"].nunique()}')
    print(f'Unique GPUs: {df_all["gpu_type"].nunique()}')

## 2. 特征工程

基于 *Watt Counts* 论文中的方法，构造以下工程特征：
- **带宽延迟**：将所有FP16权重从GPU显存读取一次的时间
- **计算延迟**：单Token理论最低计算时间
- **对数变换**：对跨度大的变量做log变换
- **加速比**：boost / base clock
- **每参数带宽**：归一化的内存带宽

In [ ]:
def engineer_features(df):
    """Engineer features based on Watt Counts paper methodology."""
    df = df.copy()
    
    if 'boost_clock_mhz' in df.columns and 'base_clock_mhz' in df.columns:
        df['boost_ratio'] = df['boost_clock_mhz'] / df['base_clock_mhz'].replace(0, np.nan)
    
    if 'total_b_params' in df.columns:
        df['log_total_b_params'] = np.log(df['total_b_params'].clip(lower=0.001))
    
    if 'total_b_params' in df.columns and 'memory_bandwidth_gb_s' in df.columns:
        bw = df['memory_bandwidth_gb_s'].clip(lower=0.01)
        df['bandwidth_latency'] = 2.0 * df['total_b_params'] / bw
    
    if 'total_b_params' in df.columns and 'tflops_16b' in df.columns:
        tflops = df['tflops_16b'].clip(lower=0.01)
        df['compute_latency_s'] = (2.0 * df['total_b_params'] * 1e9) / (tflops * 1e12)
    
    if 'total_b_params' in df.columns and 'memory_size_gb' in df.columns:
        mem = df['memory_size_gb'].clip(lower=0.01)
        df['memory_utilization'] = (df['total_b_params'] * 2.0) / mem
    
    if 'memory_bandwidth_gb_s' in df.columns and 'total_b_params' in df.columns:
        params = df['total_b_params'].clip(lower=0.001)
        df['bw_per_param'] = df['memory_bandwidth_gb_s'] / params
    
    if 'num_attention_heads' in df.columns and 'num_key_value_heads' in df.columns:
        kv = df['num_key_value_heads'].clip(lower=1)
        df['gqa_ratio'] = df['num_attention_heads'] / kv
    
    if 'hidden_size' in df.columns and 'num_layers' in df.columns:
        layers = df['num_layers'].clip(lower=1)
        df['hidden_per_layer'] = df['hidden_size'] / layers
    
    if 'intermediate_size' in df.columns and 'hidden_size' in df.columns:
        hs = df['hidden_size'].clip(lower=1)
        df['ffn_ratio'] = df['intermediate_size'] / hs
    
    if 'avg_prompt_tokens' in df.columns and 'avg_generation_tokens' in df.columns:
        df['total_tokens_per_request'] = df['avg_prompt_tokens'] + df['avg_generation_tokens']
        df['prompt_gen_ratio'] = df['avg_prompt_tokens'] / df['avg_generation_tokens'].clip(lower=1)
    
    return df

df_fe = engineer_features(df_all)

original_cols = set(df_all.columns)
new_cols = set(df_fe.columns) - original_cols
print(f'Engineered {len(new_cols)} new features:')
for c in sorted(new_cols):
    print(f'  - {c}')

## 3. 数据分割

使用简单的随机分割策略，与 dataset-split.py 保持一致。

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Use full training data; validation from pre-split A/B test sets
df_train = df_fe.copy()  # all training data
# Load pre-split test sets from Contestant Data
df_val_a = pd.read_csv('A.csv')
df_val_b = pd.read_csv('B.csv')

print(f'Train:  {len(df_train)} rows')
print(f'Val A:  {len(df_val_a)} rows (from A.csv)')
print(f'Val B:  {len(df_val_b)} rows (from B.csv)')


## 4. 模型训练

策略：为每个目标独立训练 XGBoost 模型，使用 5 折交叉验证评估。

In [ ]:
EXCLUDE_COLS = set(TARGET_COLS + ['model', 'gpu_type', 'gpu_name', 'scenario'])

feature_cols = [c for c in df_fe.columns 
                if c not in EXCLUDE_COLS 
                and df_fe[c].dtype in [np.float64, np.int64, np.int32, 'object']]

numeric_features = [c for c in feature_cols 
                    if df_fe[c].dtype in [np.float64, np.int64, np.int32]]
categorical_features = [c for c in feature_cols 
                        if df_fe[c].dtype == 'object']

print(f'Total features: {len(feature_cols)}')
print(f'  Numeric:     {len(numeric_features)}')
print(f'  Categorical: {len(categorical_features)} -> {categorical_features}')

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

print('Preprocessor ready.')

In [ ]:
X_train = df_train[feature_cols]
y_train = df_train[TARGET_COLS]

valid_mask = y_train.notna().all(axis=1)
X_train = X_train[valid_mask]
y_train = y_train[valid_mask]

print(f'Training samples: {len(X_train)}')

### 4.1 交叉验证评估

使用 5 折 KFold 评估模型泛化能力。

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

xgb_params = {
    'n_estimators': 200,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 10,
    'reg_alpha': 1,
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
}

cv_predictions = pd.DataFrame(index=X_train.index, columns=TARGET_COLS)

for target_idx, target in enumerate(TARGET_COLS):
    print(f'\n--- Training for target: {target} ---')
    
    y_target = y_train[target].values
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_target[train_idx], y_target[val_idx]
        
        X_tr_p = preprocessor.fit_transform(X_tr)
        X_val_p = preprocessor.transform(X_val)
        
        model = XGBRegressor(**xgb_params)
        model.fit(X_tr_p, y_tr, verbose=False)
        
        cv_predictions.iloc[val_idx, target_idx] = model.predict(X_val_p)
        
        mae = mean_absolute_error(y_val, cv_predictions.iloc[val_idx, target_idx])
        print(f'  Fold {fold+1}: MAE = {mae:.4f}')

In [ ]:
print('\n' + '=' * 60)
print('Cross-Validation Results')
print('=' * 60)

overall_wmape = []
overall_r2 = []

for i, target in enumerate(TARGET_COLS):
    y_true = y_train[target].values
    y_pred = cv_predictions[target].values
    valid = ~pd.isna(y_pred)
    y_true_v = y_true[valid]
    y_pred_v = y_pred[valid]
    
    mae = mean_absolute_error(y_true_v, y_pred_v)
    r2 = r2_score(y_true_v, y_pred_v)
    wmape = np.sum(np.abs(y_true_v - y_pred_v)) / np.sum(np.abs(y_true_v)) * 100
    
    overall_wmape.append(wmape)
    overall_r2.append(r2)
    
    print(f'\n{target}:')
    print(f'  MAE:   {mae:.4f}')
    print(f'  R2:    {r2:.4f}')
    print(f'  WMAPE: {wmape:.2f}%')

print(f'\nAverage WMAPE: {np.mean(overall_wmape):.2f}%')
print(f'Average WMAPE (Aux): {np.mean(overall_wmape):.2f}%')

### 4.2 训练最终模型

In [ ]:
final_models = {}
X_train_p = preprocessor.fit_transform(X_train)

for target in TARGET_COLS:
    print(f'Training final model for: {target}')
    y_target = y_train[target].values
    
    model = XGBRegressor(**xgb_params)
    model.fit(X_train_p, y_target, verbose=False)
    final_models[target] = model

print('\nAll final models trained!')

## 5. 预测与验证

In [ ]:
def predict_and_evaluate(df_test, name):
    """Predict on test set and evaluate against ground truth."""
    df_test_fe = engineer_features(df_test)
    X_test = df_test_fe[feature_cols].copy()
    X_test_p = preprocessor.transform(X_test)
    
    predictions = {}
    for target in TARGET_COLS:
        predictions[target] = final_models[target].predict(X_test_p)
    
    df_pred = pd.DataFrame(predictions)
    pred_path = f'{name}_predict.csv'
    df_pred.to_csv(pred_path, index=False)
    print(f'Saved: {pred_path}')

In [ ]:
pred_a = predict_and_evaluate(df_val_a, 'A')

In [ ]:
pred_b = predict_and_evaluate(df_val_b, 'B')

## 6. 特征重要性分析

In [ ]:
print('Top 10 features for each target:')
for target in TARGET_COLS:
    if target in final_models:
        try:
            names = preprocessor.get_feature_names_out()
            names = [n.replace('num__', '').replace('cat__', '') for n in names]
        except:
            names = [f'f{i}' for i in range(len(final_models[target].feature_importances_))]
        
        imp = pd.DataFrame({
            'feature': names,
            'importance': final_models[target].feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f'\n--- {target} ---')
        print(imp.head(10).to_string(index=False))